In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q tensorflow nltk rouge-score

  Preparing metadata (setup.py) ... done


In [58]:
!pip install rouge-score

import os
import ast
import pickle
import warnings
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

# Suppress warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')
warnings.filterwarnings('ignore', category=UserWarning, module='keras')

# Download NLTK requirements
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

# --- CONFIGURATION ---
CONFIG = {
    'MODEL_PATH': 'model_19.h5',
    'WORD_TO_IDX_PATH': 'word_to_idx.pkl',
    'IDX_TO_WORD_PATH': 'idx_to_word.pkl',
    'ENCODED_FEATURES_PATH': 'encoded_test_features.pkl',
    'TEST_IMAGES_FILE': 'testImages.txt',
    'TOKENS_CLEAN_FILE': 'tokens_clean.txt',
    'MAX_LEN': 38
}

print("Checking files...")
missing_files = [f for f in CONFIG.values() if isinstance(f, str) and not os.path.exists(f)]
if missing_files:
    print(f"CRITICAL ERROR: Missing files in Colab! Please upload: {missing_files}")
    raise FileNotFoundError(f"Upload these files before continuing: {missing_files}")

# --- LOAD DATA & MODELS ---
print('Loading Vocabularies...')
with open(CONFIG['WORD_TO_IDX_PATH'], 'rb') as f:
    word_to_idx = pickle.load(f)
with open(CONFIG['IDX_TO_WORD_PATH'], 'rb') as f:
    idx_to_word = pickle.load(f)

print('Loading caption model...')
caption_model = load_model(CONFIG['MODEL_PATH'], compile=False)

print('Loading pre-encoded features...')
with open(CONFIG['ENCODED_FEATURES_PATH'], 'rb') as f:
    raw_features = pickle.load(f)
    encoded_features = {k.replace('.jpg', ''): v for k, v in raw_features.items()}

# --- HELPER FUNCTIONS ---
smoothing_fn = SmoothingFunction().method1
r_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def predict_caption(photo, model, word_to_idx, idx_to_word, max_len):
    in_text = 'startseq'
    for _ in range(max_len):
        seq = [word_to_idx[w] for w in in_text.split() if w in word_to_idx]
        if not seq:
            break
        sequence = pad_sequences([seq], maxlen=max_len, padding='post')
        yhat = model.predict([photo, sequence], verbose=0)
        yhat = np.argmax(yhat)
        word = idx_to_word.get(yhat, '')
        if not word:
            break
        in_text += ' ' + word
        if word == 'endseq':
            break
    return ' '.join([w for w in in_text.split() if w not in {'startseq', 'endseq'}])

def compute_metrics(references, hypothesis):
    # Pre-tokenize for NLTK compatibility
    ref_tokens = [nltk.word_tokenize(r.lower()) for r in references]
    hyp_tokens = nltk.word_tokenize(hypothesis.lower())
    rouge_l = r_scorer.score(' '.join(references), hypothesis)['rougeL'].fmeasure

    return {
        'bleu1': sentence_bleu(ref_tokens, hyp_tokens, weights=(1, 0, 0, 0), smoothing_function=smoothing_fn),
        'bleu2': sentence_bleu(ref_tokens, hyp_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothing_fn),
        'bleu3': sentence_bleu(ref_tokens, hyp_tokens, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoothing_fn),
        'bleu4': sentence_bleu(ref_tokens, hyp_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing_fn),
        'meteor': meteor_score(ref_tokens, hyp_tokens), # Fixed NLTK pre-tokenized requirement!
        'rouge_l': rouge_l,
        'exact_match': 1.0 if hypothesis.strip().lower() in [r.strip().lower() for r in references] else 0.0,
    }

# --- EVALUATION SCRIPT ---
print("\nLoading Evaluation Sets...")
with open(CONFIG['TOKENS_CLEAN_FILE'], 'r', encoding='utf-8') as f:
    references = ast.literal_eval(f.read())  # Load all dictionary references

with open(CONFIG['TEST_IMAGES_FILE'], 'r', encoding='utf-8') as f:
    test_ids = [line.strip().replace('.jpg', '') for line in f if line.strip()]

# LIMIT EVALUATION TO 200 IMAGES
test_ids = test_ids[:200]

print(f"-> Total Ground Truth captions: {len(references)}")
print(f"-> Test Images to Evaluate: {len(test_ids)}")

metrics_sum = {k: 0.0 for k in ['bleu1', 'bleu2', 'bleu3', 'bleu4', 'meteor', 'rouge_l', 'exact_match']}
count = 0

print("\nStarting Evaluation...")
for img_id in test_ids:
    if img_id not in references or img_id not in encoded_features:
        continue

    photo = np.reshape(np.array(encoded_features[img_id]), (1, -1))
    pred = predict_caption(photo, caption_model, word_to_idx, idx_to_word, CONFIG['MAX_LEN'])
    scores = compute_metrics(references[img_id], pred)

    for k in metrics_sum:
        metrics_sum[k] += scores[k]
    count += 1

    if count % 20 == 0:
        print(f"Processed {count}/{len(test_ids)} images...")

# --- RESULTS ---
if count > 0:
    print('\n' + '='*35)
    print('        FINAL TEST SCORES')
    print('='*35)
    for k, v in metrics_sum.items():
        print(f"{k.upper():12}: {v/count:.4f}")
    print('='*35)
else:
    print("WARNING: No images processed. Ensure image IDs match perfectly.")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Checking files...
Loading Vocabularies...
Loading caption model...
Loading pre-encoded features...

Loading Evaluation Sets...
-> Total Ground Truth captions: 8092
-> Test Images to Evaluate: 200

Starting Evaluation...
Processed 20/200 images...
Processed 40/200 images...
Processed 60/200 images...
Processed 80/200 images...
Processed 100/200 images...
Processed 120/200 images...
Processed 140/200 images...
Processed 160/200 images...
Processed 180/200 images...
Processed 200/200 images...

        FINAL TEST SCORES
BLEU1       : 0.5359
BLEU2       : 0.4532
BLEU3       : 0.3131
BLEU4       : 0.2111
METEOR      : 0.3280
ROUGE_L     : 0.1751
